# Sovereign Debt Distress — Data Cleaning and Feature Engineering

This notebook takes the merged base panel (`data/processed/panel_full.csv`) and the raw FRED and commodity terms-of-trade series, applies the full feature engineering pipeline, and writes the final train / validation / test splits used by the model runner.

**Outputs written to `data/final/`:**
- `train_flat.csv`, `val_flat.csv`, `test_flat.csv` — flat feature tables
- `train_windows.npz`, `val_windows.npz`, `test_windows.npz` — 24-month sliding windows
- `train_meta.csv`, `val_meta.csv`, `test_meta.csv` — row-level metadata
- `feature_metadata.json`, `class_distribution.json`, `summary_statistics.csv`

**Time splits:**
- Train: Jan 2000 – Dec 2017
- Embargo: Jan 2018 – Dec 2018 (not used for training or evaluation)
- Validation: Jan 2019 – Dec 2020
- Test: Jan 2022 – Dec 2024


In [ ]:
# 1. Setup

import os, json, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 180)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
# 2. Set paths

from pathlib import Path

# Set DATA_DIR to the root of the data directory in this repository.
# If running from notebooks/, use the default below.
# Adjust if running from a different working directory.
DATA_DIR = Path("../data")

# Optional: if you uploaded DATA.zip and want to extract it first.
ZIP_PATH = None  # e.g. Path("../data.zip")

import zipfile
if ZIP_PATH is not None and ZIP_PATH.exists() and not DATA_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(ZIP_PATH.parent)

if (DATA_DIR / "DATA").exists() and not (DATA_DIR / "raw").exists():
    DATA_DIR = DATA_DIR / "DATA"

print("DATA_DIR:", DATA_DIR.resolve())
print("Exists:", DATA_DIR.exists())


In [ ]:
# 3. Validate required files

def rename_if_exists(src, dst):
    src, dst = Path(src), Path(dst)
    if src.exists() and not dst.exists():
        print(f"Renaming {src} -> {dst}")
        src.rename(dst)

rename_if_exists(DATA_DIR / "Processed", DATA_DIR / "processed")
rename_if_exists(DATA_DIR / "raw" / "imf dots", DATA_DIR / "raw" / "imf_dots")
rename_if_exists(DATA_DIR / "raw" / "imf-dots", DATA_DIR / "raw" / "imf_dots")

(DATA_DIR / "interim").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "final").mkdir(parents=True, exist_ok=True)

required = [
    DATA_DIR / "processed" / "panel_full.csv",
    DATA_DIR / "raw" / "fred" / "VIXCLS.csv",
    DATA_DIR / "raw" / "fred" / "DGS10.csv",
    DATA_DIR / "raw" / "fred" / "FEDFUNDS.csv",
    DATA_DIR / "raw" / "fred" / "DCOILBRENTEU.csv",
    DATA_DIR / "raw" / "fred" / "CPIAUCSL.csv",
    DATA_DIR / "raw" / "fred" / "TWEXBMTH.csv",
    DATA_DIR / "raw" / "fred" / "DTWEXBGS.csv",
    DATA_DIR / "raw" / "ctot" / "ctot_model_input_wide_2000_2024.csv",
    DATA_DIR / "raw" / "imf_dots" / "imports_monthly.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print("Missing files:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Fix missing files before continuing.")

print("All required files found.")

In [ ]:
# 4. Build curated distress labels

panel_base = pd.read_csv(DATA_DIR / "processed" / "panel_full.csv", usecols=["iso3", "year_month"])
panel_base["year_month"] = pd.to_datetime(panel_base["year_month"])
countries = sorted(panel_base["iso3"].dropna().unique())
panel_dates = pd.date_range(panel_base["year_month"].min(), panel_base["year_month"].max(), freq="MS")

curated = pd.DataFrame([
    ("ARG", "2001-11-01", "Sovereign default"),
    ("ARG", "2014-07-01", "Technical default"),
    ("ARG", "2020-05-01", "Debt restructuring"),
    ("ECU", "2008-12-01", "Voluntary default"),
    ("ECU", "2020-04-01", "Restructuring"),
    ("VEN", "2017-11-01", "Bond default"),
    ("DOM", "2005-04-01", "Restructuring"),
    ("JAM", "2010-02-01", "JDX debt exchange"),
    ("JAM", "2013-02-01", "NDX debt exchange"),
    ("TUR", "2001-02-01", "Banking crisis + IMF"),
    ("UKR", "2015-09-01", "Eurobond restructuring"),
    ("UKR", "2022-01-01", "War + payment suspension"),
    ("MOZ", "2016-04-01", "Hidden debt crisis"),
    ("LBN", "2020-03-01", "Eurobond default"),
    ("ZMB", "2020-11-01", "Eurobond default"),
    ("LKA", "2022-05-01", "Sovereign default"),
    ("GHA", "2022-12-01", "Debt restructuring"),
    ("ETH", "2023-12-01", "Common Framework"),
    ("PAK", "2023-01-01", "Near-default / IMF"),
    ("EGY", "2016-11-01", "IMF emergency"),
    ("KEN", "2024-07-01", "Debt stress / IMF"),
    ("TUN", "2013-06-01", "IMF SBA"),
], columns=["iso3", "start_month", "event_type"])

curated["start_month"] = pd.to_datetime(curated["start_month"])
curated = curated[curated["iso3"].isin(countries)].copy()
curated.to_csv(DATA_DIR / "interim" / "distress_events_curated.csv", index=False)

rows = []
for iso3 in countries:
    evs = curated.loc[curated["iso3"] == iso3, "start_month"].to_numpy()
    for ym in panel_dates:
        y = int(any((ev > ym) and (ev <= ym + pd.DateOffset(months=12)) for ev in evs))
        rows.append({"iso3": iso3, "year_month": ym, "distress_within_12m": y})

labels = pd.DataFrame(rows)
labels.to_csv(DATA_DIR / "interim" / "labels_curated_monthly.csv", index=False)

print("Curated events:", len(curated))
print("Labels:", labels.shape)
print("Total y=1:", int(labels["distress_within_12m"].sum()))
print("Positive rate:", f"{labels['distress_within_12m'].mean():.3%}")
display(labels.groupby("iso3")["distress_within_12m"].sum().sort_values(ascending=False).head(20))

In [ ]:
# 5. Default-history features

curated = pd.read_csv(DATA_DIR / "interim" / "distress_events_curated.csv", parse_dates=["start_month"])

rows = []
for iso3 in countries:
    evs = sorted(curated.loc[curated["iso3"] == iso3, "start_month"].tolist())
    for ym in panel_dates:
        past = [e for e in evs if e < ym]
        rows.append({
            "iso3": iso3,
            "year_month": ym,
            "total_past_defaults": len(past),
            "has_defaulted_before": int(len(past) > 0),
            "years_since_last_default": round((ym - max(past)).days / 365.25, 2) if past else 99.0,
            "currently_in_default": int(any((ym >= e) and (ym < e + pd.DateOffset(months=12)) for e in evs)),
        })

default_hist = pd.DataFrame(rows)
default_hist.to_csv(DATA_DIR / "interim" / "default_history_monthly.csv", index=False)
print("Default history:", default_hist.shape)

In [ ]:
# 6. FRED global macro features

FRED_DIR = DATA_DIR / "raw" / "fred"

def read_fred(path, value_col):
    df = pd.read_csv(path)
    df.columns = [str(c).strip().replace("\ufeff", "") for c in df.columns]
    if "observation_date" not in df.columns:
        df = df.rename(columns={df.columns[0]: "observation_date"})
    if value_col not in df.columns:
        candidates = [c for c in df.columns if c != "observation_date"]
        df = df.rename(columns={candidates[0]: value_col})
    df["observation_date"] = pd.to_datetime(df["observation_date"], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    return df[["observation_date", value_col]].dropna(subset=["observation_date"])

def to_monthly(df, value_col):
    return (
        df.set_index("observation_date")
          .sort_index()[value_col]
          .resample("MS")
          .last()
          .reset_index()
          .rename(columns={"observation_date": "year_month"})
    )

vix_m = to_monthly(read_fred(FRED_DIR / "VIXCLS.csv", "VIXCLS"), "VIXCLS").rename(columns={"VIXCLS": "global_vix"})
dgs_m = to_monthly(read_fred(FRED_DIR / "DGS10.csv", "DGS10"), "DGS10").rename(columns={"DGS10": "global_us_10y"})
brent_m = to_monthly(read_fred(FRED_DIR / "DCOILBRENTEU.csv", "DCOILBRENTEU"), "DCOILBRENTEU").rename(columns={"DCOILBRENTEU": "global_brent_oil"})
dxy_new_m = to_monthly(read_fred(FRED_DIR / "DTWEXBGS.csv", "DTWEXBGS"), "DTWEXBGS").rename(columns={"DTWEXBGS": "global_dxy"})

ff_m = read_fred(FRED_DIR / "FEDFUNDS.csv", "FEDFUNDS").rename(columns={"observation_date": "year_month", "FEDFUNDS": "global_fed_funds"})
cpi_m = read_fred(FRED_DIR / "CPIAUCSL.csv", "CPIAUCSL").rename(columns={"observation_date": "year_month", "CPIAUCSL": "us_cpi"})
dxy_old_m = read_fred(FRED_DIR / "TWEXBMTH.csv", "TWEXBMTH").rename(columns={"observation_date": "year_month", "TWEXBMTH": "global_dxy_old"})

merged_dxy = dxy_old_m.merge(dxy_new_m, on="year_month", how="inner")
if len(merged_dxy) > 0:
    scale = merged_dxy["global_dxy"].mean() / merged_dxy["global_dxy_old"].mean()
    dxy_old_m["global_dxy_scaled"] = dxy_old_m["global_dxy_old"] * scale
    dxy_spliced = pd.concat([
        dxy_old_m[dxy_old_m["year_month"] < "2006-01-01"][["year_month", "global_dxy_scaled"]].rename(columns={"global_dxy_scaled": "global_dxy"}),
        dxy_new_m[dxy_new_m["year_month"] >= "2006-01-01"]
    ]).sort_values("year_month")
else:
    dxy_spliced = dxy_new_m

fred = vix_m.merge(dgs_m, on="year_month", how="outer")
fred = fred.merge(ff_m, on="year_month", how="outer")
fred = fred.merge(brent_m, on="year_month", how="outer")
fred = fred.merge(dxy_spliced, on="year_month", how="outer")
fred = fred.merge(cpi_m, on="year_month", how="outer")
fred = fred[(fred["year_month"] >= "2000-01-01") & (fred["year_month"] <= "2024-12-01")]
fred = fred.sort_values("year_month").reset_index(drop=True)

fred["global_vix_3m_change"] = fred["global_vix"].diff(3)
fred["global_us_10y_3m_change"] = fred["global_us_10y"].diff(3)
fred["global_us_10y_6m_change"] = fred["global_us_10y"].diff(6)
fred["global_fed_funds_6m_change"] = fred["global_fed_funds"].diff(6)
fred["global_fed_funds_12m_change"] = fred["global_fed_funds"].diff(12)
fred["global_dxy_3m_pct_change"] = fred["global_dxy"].pct_change(3) * 100
fred["global_dxy_6m_pct_change"] = fred["global_dxy"].pct_change(6) * 100
fred["global_brent_3m_pct_change"] = fred["global_brent_oil"].pct_change(3) * 100
fred["global_brent_6m_pct_change"] = fred["global_brent_oil"].pct_change(6) * 100
fred["global_yield_curve"] = fred["global_us_10y"] - fred["global_fed_funds"]
fred["us_cpi_yoy"] = fred["us_cpi"].pct_change(12) * 100
fred["global_real_rate"] = fred["global_fed_funds"] - fred["us_cpi_yoy"]
fred["global_tightening_dummy"] = (fred["global_fed_funds_12m_change"] > 1.0).astype(int)

for c in ["global_vix", "global_dxy", "global_us_10y"]:
    fred[f"{c}_pctile"] = fred[c].rank(pct=True)

fred["global_risk_composite"] = fred[["global_vix_pctile", "global_dxy_pctile", "global_us_10y_pctile"]].mean(axis=1)
fred = fred.drop(columns=["us_cpi", "us_cpi_yoy", "global_vix_pctile", "global_dxy_pctile", "global_us_10y_pctile"], errors="ignore")
fred.to_csv(DATA_DIR / "interim" / "fred_global_monthly.csv", index=False)

print("FRED:", fred.shape)
display(fred[fred["year_month"] == "2020-03-01"][["year_month", "global_vix"]])

In [ ]:
# 7. Commodity Terms of Trade

ctot = pd.read_csv(DATA_DIR / "raw" / "ctot" / "ctot_model_input_wide_2000_2024.csv")
ctot.columns = [str(c).strip().replace("\ufeff", "") for c in ctot.columns]

ctot_map = {
    "Argentina": "ARG", "Brazil": "BRA", "Colombia": "COL", "Ecuador": "ECU",
    "Ghana": "GHA", "Indonesia": "IDN", "Jordan": "JOR", "Kenya": "KEN",
    "Lebanon": "LBN", "Sri Lanka": "LKA", "Mexico": "MEX", "Malaysia": "MYS",
    "Nigeria": "NGA", "Peru": "PER", "Philippines": "PHL", "Pakistan": "PAK",
    "Russian Federation": "RUS", "Russia": "RUS", "Tunisia": "TUN",
    "Ukraine": "UKR", "Zambia": "ZMB", "South Africa": "ZAF",
}
ctot = ctot[ctot["COUNTRY"].isin(ctot_map.keys())].copy()
ctot["iso3"] = ctot["COUNTRY"].map(ctot_map)

date_cols = [c for c in ctot.columns if "-M" in str(c)]
ctot_long = ctot.melt(id_vars=["iso3"], value_vars=date_cols, var_name="period", value_name="ctot_index")
ctot_long["year_month"] = pd.to_datetime(ctot_long["period"].astype(str).str.replace("-M", "-", regex=False), format="%Y-%m", errors="coerce")
ctot_long["ctot_index"] = pd.to_numeric(ctot_long["ctot_index"], errors="coerce")
ctot_long = ctot_long.drop(columns=["period"]).dropna(subset=["year_month"]).sort_values(["iso3", "year_month"]).reset_index(drop=True)

for iso3, grp in ctot_long.groupby("iso3"):
    idx = grp.index
    ctot_long.loc[idx, "ctot_yoy_change"] = grp["ctot_index"].pct_change(12) * 100
    ctot_long.loc[idx, "ctot_6m_change"] = grp["ctot_index"].pct_change(6) * 100
    ctot_long.loc[idx, "ctot_below_95"] = (grp["ctot_index"] < 95).astype(int)

ctot_long.to_csv(DATA_DIR / "interim" / "commodity_tot_monthly.csv", index=False)
print("CToT:", ctot_long.shape, "countries:", ctot_long["iso3"].nunique())

In [ ]:
# 8. Merge into final panel

panel = pd.read_csv(DATA_DIR / "processed" / "panel_full.csv", parse_dates=["year_month"])
labels = pd.read_csv(DATA_DIR / "interim" / "labels_curated_monthly.csv", parse_dates=["year_month"])
default_hist = pd.read_csv(DATA_DIR / "interim" / "default_history_monthly.csv", parse_dates=["year_month"])
fred = pd.read_csv(DATA_DIR / "interim" / "fred_global_monthly.csv", parse_dates=["year_month"])
ctot = pd.read_csv(DATA_DIR / "interim" / "commodity_tot_monthly.csv", parse_dates=["year_month"])

old_label_cols = ["distress_within_12m", "in_distress_now", "months_to_next_distress"]
old_missing_cols = [c for c in panel.columns if c.endswith("_missing")]
panel = panel.drop(columns=old_label_cols + old_missing_cols, errors="ignore")

panel = panel.merge(labels, on=["iso3", "year_month"], how="left")
panel = panel.merge(default_hist, on=["iso3", "year_month"], how="left")
panel = panel.merge(fred, on="year_month", how="left")
panel = panel.merge(ctot, on=["iso3", "year_month"], how="left")

panel["distress_within_12m"] = panel["distress_within_12m"].fillna(0).astype(int)

# Explicitly exclude unreliable liquidity / reserves-imports features from final modeling.
bad_liquidity_cols = [
    "reserves_months_imports",
    "reserves_months_imports_6m_change",
    "reserves_months_imports_below_3",
    "reserves_months_imports_missing",
    "reserves_months_imports_6m_change_missing",
]
panel = panel.drop(columns=bad_liquidity_cols, errors="ignore")

id_cols = ["iso3", "year_month", "country_name", "year", "month_num"]
label_cols = ["distress_within_12m"]

feature_cols = [
    c for c in panel.columns
    if c not in id_cols + label_cols
    and c != "split"
    and not c.endswith("_missing")
    and pd.api.types.is_numeric_dtype(panel[c])
]

# Create missing flags, excluding any reserves/imports flags.
missing_flag_cols = []
for col in feature_cols:
    if panel[col].isna().mean() > 0.10:
        fn = f"{col}_missing"
        if fn not in bad_liquidity_cols and not fn.startswith("reserves_months_imports"):
            panel[fn] = panel[col].isna().astype(int)
            missing_flag_cols.append(fn)

panel = panel.sort_values(["iso3", "year_month"]).reset_index(drop=True)

for col in feature_cols:
    panel[col] = panel.groupby("iso3")[col].ffill()
    panel[col] = panel.groupby("iso3")[col].bfill(limit=6)
    if panel[col].isna().any():
        med = panel[col].median()
        if pd.isna(med):
            med = 0.0
        panel[col] = panel[col].fillna(med)

# Split
panel["split"] = "embargo"
panel.loc[panel["year_month"] <= "2017-12-01", "split"] = "train"
panel.loc[(panel["year_month"] >= "2019-01-01") & (panel["year_month"] <= "2020-12-01"), "split"] = "val"
# Jan 2021 – Dec 2021 remains "embargo" (second gap before test set)
panel.loc[panel["year_month"] >= "2022-01-01", "split"] = "test"

flat_features = feature_cols + missing_flag_cols

# Safety check: no bad liquidity columns in final features.
bad_found = [c for c in flat_features if c in bad_liquidity_cols or c.startswith("reserves_months_imports")]
if bad_found:
    raise ValueError(f"Bad liquidity columns still present: {bad_found}")

print("Panel after merge:", panel.shape)
print("Features:", len(feature_cols), "+ flags:", len(missing_flag_cols), "=", len(flat_features))
print("Max missing in features:", panel[feature_cols].isna().mean().max())

for s in ["train", "val", "test", "embargo"]:
    sub = panel[panel["split"] == s]
    y1 = int(sub["distress_within_12m"].sum())
    print(f"{s:8s}: {len(sub):5d} rows | y=1: {y1:4d} | rate: {y1 / len(sub) if len(sub) else 0:.2%}")

In [ ]:
# 9. Save flat CSVs and windowed NPZ files

final_dir = DATA_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)

# Save flat files
for s in ["train", "val", "test"]:
    sub = panel[panel["split"] == s].copy()
    sub[["iso3", "year_month"] + flat_features + ["distress_within_12m"]].to_csv(final_dir / f"{s}_flat.csv", index=False)
    print(f"Saved {s}_flat.csv:", sub.shape[0], "rows x", len(flat_features), "features")

# FIXED sequence generation:
# Use end-month split only. Do NOT drop windows because they overlap embargo months.
WINDOW = 24
all_X, all_y, all_meta = [], [], []
panel_sorted = panel.sort_values(["iso3", "year_month"]).reset_index(drop=True)

for iso3, grp in panel_sorted.groupby("iso3"):
    grp = grp.reset_index(drop=True)
    vals = grp[feature_cols].values.astype(np.float32)
    labs = grp["distress_within_12m"].fillna(0).astype(int).values
    splits = grp["split"].values
    months = pd.to_datetime(grp["year_month"].values)

    for end in range(WINDOW - 1, len(grp)):
        end_split = splits[end]
        if end_split == "embargo":
            continue

        start = end - WINDOW + 1
        all_X.append(vals[start:end + 1])
        all_y.append(int(labs[end]))
        all_meta.append({
            "iso3": iso3,
            "window_end": str(pd.Timestamp(months[end]).date()),
            "y": int(labs[end]),
            "split": end_split
        })

all_X = np.stack(all_X)
all_y = np.array(all_y, dtype=np.int64)
meta_df = pd.DataFrame(all_meta)

for s in ["train", "val", "test"]:
    mask = meta_df["split"] == s
    X_s = all_X[mask.values]
    y_s = all_y[mask.values]
    np.savez_compressed(final_dir / f"{s}_windows.npz", X=X_s, y=y_s)
    meta_df[mask].to_csv(final_dir / f"{s}_meta.csv", index=False)
    print(f"{s:5s} windows: X={X_s.shape} | y=1: {(y_s == 1).sum()}")

# Save full panel
panel.to_csv(final_dir / "panel_full.csv", index=False)

In [ ]:
# 10. Save metadata and summary statistics

modality_map = {
    "macro": [c for c in feature_cols if any(c.startswith(p) for p in ["cpi_", "fx_", "weo_", "ir_"])],
    "debt": [c for c in feature_cols if any(c.startswith(p) for p in ["edt_", "bis_", "short_term_debt", "debt_to_gni", "debt_stress"])],
    "political": [c for c in feature_cols if any(c.startswith(p) for p in ["acled_", "political_violence", "instability_x", "fatalities_x", "violence_per"])],
    "bop": [c for c in feature_cols if c.startswith("bop_") or c == "capital_flow_stress"],
    "global": [c for c in feature_cols if c.startswith("global_")],
    "structural": [c for c in feature_cols if any(c.startswith(p) for p in ["years_since", "total_past", "has_defaulted", "currently_in_default"])],
    "commodity": [c for c in feature_cols if c.startswith("ctot_")],
    "liquidity": [],  # intentionally removed until a true reserves-stock variable is available
}

metadata = {
    "feature_columns": feature_cols,
    "missing_flags": missing_flag_cols,
    "flat_features": flat_features,
    "dropped_unreliable_columns": [
        "reserves_months_imports",
        "reserves_months_imports_6m_change",
        "reserves_months_imports_below_3",
        "reserves_months_imports_missing",
        "reserves_months_imports_6m_change_missing",
    ],
    "window_size": 24,
    "window_generation_fix": "windows assigned by end-month split only; embargo overlap is not used to drop windows",
    "modality_map": modality_map,
    "n_features": len(feature_cols),
    "n_flat_features": len(flat_features),
    "n_countries": int(panel["iso3"].nunique()),
}

with open(final_dir / "feature_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

class_distribution = {}
for s in ["train", "val", "test"]:
    sub = panel[panel["split"] == s]
    y1 = int(sub["distress_within_12m"].sum())
    class_distribution[s] = {
        "y1": y1,
        "y0": int(len(sub) - y1),
        "total": int(len(sub)),
        "rate": round(y1 / len(sub), 4) if len(sub) else None,
    }

with open(final_dir / "class_distribution.json", "w") as f:
    json.dump(class_distribution, f, indent=2)

panel[feature_cols].describe().T.to_csv(final_dir / "summary_statistics.csv")

print("=" * 70)
print("FINAL FIXED DATASET COMPLETE")
print("=" * 70)
print("Output folder:", final_dir)
print("Panel:", panel.shape)
print("Flat features:", len(flat_features))

print("\nModalities:")
for k, v in modality_map.items():
    print(f"{k:12s}: {len(v)}")

print("\nClass distribution:")
display(pd.DataFrame(class_distribution).T)

In [ ]:
# 11. Zip outputs

zip_path = shutil.make_archive(str(DATA_DIR / "final_outputs"), "zip", final_dir)
print("Created:", zip_path)


Final outputs are saved in:

```text
data/final/
```

Zipped output:

```text
data/final_outputs.zip
```